# MIMIC-IV Sepsis-3 Cohort Selection

This notebook selects patients meeting Sepsis-3 criteria with specific inclusion/exclusion criteria.

**Purpose:** Create a research-ready cohort for Sepsis-3 analysis

**Dataset:** `my-new-project-473015.my_mimiciv_derived`

---

## Research Design: Cohort Selection Criteria

### Inclusion Criteria
1. ✅ Diagnosis with Sepsis-3 criteria (from `my_mimiciv_derived.sepsis3`)
2. ✅ First ICU admission with sepsis diagnosis
3. ✅ Age ≥18 years

### Exclusion Criteria
1. ❌ ICU discharge within 24 hours

---

## 📊 Flow Diagram Importance

**Why Flow Diagrams Matter in Research:**

A CONSORT-style flow diagram is essential for:
- **Transparency**: Shows exactly how many patients were excluded at each step
- **Reproducibility**: Others can replicate your cohort selection
- **Quality Control**: Identifies potential data quality issues
- **Publication Requirements**: Most journals require flow diagrams for cohort studies

We will track patient counts at each step to create a complete flow diagram.

---

## Analysis Steps

**STEP 0:** Setup and configuration  
**STEP 1:** Extract all Sepsis-3 patients  
**STEP 2:** Filter for first ICU admission with sepsis  
**STEP 3:** Apply age filter (≥18 years)  
**STEP 4:** Exclude short ICU stays (<24 hours) - Compare two methods  
**STEP 5:** Create final cohort table  
**STEP 6:** Generate flow diagram and cohort statistics

Let's begin!

---

## STEP 0: Setup

In [1]:
# Import libraries
from google.cloud import bigquery
import pandas as pd
from datetime import datetime
import matplotlib.pyplot as plt

# Initialize BigQuery client
client = bigquery.Client(project='my-new-project-473015')

# Configuration
PROJECT_ID = 'my-new-project-473015'
DATASET_ID = 'my_mimiciv_derived'

print("="*70)
print("MIMIC-IV Sepsis-3 Cohort Selection")
print(f"Target dataset: {PROJECT_ID}.{DATASET_ID}")
print(f"Timestamp: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print("="*70)
print("\n✅ Setup complete! Ready to select cohort.")
print("\n📊 We will track patient counts at each step for the flow diagram.")

MIMIC-IV Sepsis-3 Cohort Selection
Target dataset: my-new-project-473015.my_mimiciv_derived
Timestamp: 2025-10-09 17:15:32

✅ Setup complete! Ready to select cohort.

📊 We will track patient counts at each step for the flow diagram.


---

## STEP 1: Extract All Sepsis-3 Patients

Extract all patients diagnosed with Sepsis-3 from our previously created table.

**Criteria:** `sepsis3 = TRUE`

**What we extract:**
- Patient identifiers (subject_id, stay_id)
- Sepsis onset times
- SOFA scores at onset
- Organ dysfunction components

This is our **starting population** for the flow diagram.

In [3]:
%%bigquery step1_result

SELECT
    subject_id,
    stay_id,
    suspected_infection_time,
    sofa_time,
    sofa_score,
    respiration,
    coagulation,
    liver,
    cardiovascular,
    cns,
    renal,
    sepsis3
FROM `my-new-project-473015.my_mimiciv_derived.sepsis3`
WHERE sepsis3 = TRUE;

Query is running:   0%|          |

Downloading:   0%|          |

In [5]:

# Count patients
step1_count = len(step1_result)
step1_unique_patients = step1_result['subject_id'].nunique()
step1_unique_stays = step1_result['stay_id'].nunique()

print("="*70)
print("STEP 1: All Sepsis-3 Patients")
print("="*70)
print(f"Total rows (sepsis events): {step1_count:,}")
print(f"Unique patients: {step1_unique_patients:,}")
print(f"Unique ICU stays: {step1_unique_stays:,}")
print("\n📊 Flow Diagram - Starting Population:")
print(f"   N = {step1_unique_stays:,} ICU stays with Sepsis-3")
print("="*70)

# Show sample
print("\n📋 Sample data (first 5 rows):")
print(step1_result.head())

STEP 1: All Sepsis-3 Patients
Total rows (sepsis events): 43,705
Unique patients: 33,311
Unique ICU stays: 43,705

📊 Flow Diagram - Starting Population:
   N = 43,705 ICU stays with Sepsis-3

📋 Sample data (first 5 rows):
   subject_id   stay_id suspected_infection_time           sofa_time  \
0    10146603  35119940      2199-01-14 20:00:00 2199-01-14 19:00:00   
1    18175636  32937083      2129-10-27 07:26:00 2129-10-27 08:00:00   
2    10144145  39712677      2115-01-02 20:56:00 2115-01-02 21:00:00   
3    19526256  30409303      2159-12-20 08:00:00 2159-12-20 08:00:00   
4    18041094  33996330      2202-12-07 16:00:00 2202-12-08 06:00:00   

   sofa_score  respiration  coagulation  liver  cardiovascular  cns  renal  \
0           2            0            0      0               2    0      0   
1           2            0            0      0               2    0      0   
2           2            0            0      0               2    0      0   
3           2            0       

---

## STEP 2: Filter for First ICU Admission with Sepsis

Select only the **first ICU admission** where sepsis was diagnosed for each patient.

**Why this matters:**
- Patients may have multiple ICU admissions
- We want to study the **first sepsis event** to avoid bias from previous treatments
- Uses `ROW_NUMBER()` to rank ICU admissions by `intime`

**Method:**
1. Join with `icustays` table to get admission times
2. Rank each patient's ICU stays by admission time
3. Select only rank = 1 (first admission with sepsis)

In [6]:
%%bigquery step2_result

WITH sepsis_patients AS (
    SELECT
        s.subject_id,
        s.stay_id,
        s.suspected_infection_time,
        s.sofa_time,
        s.sofa_score,
        s.respiration,
        s.coagulation,
        s.liver,
        s.cardiovascular,
        s.cns,
        s.renal
    FROM `my-new-project-473015.my_mimiciv_derived.sepsis3` s
    WHERE s.sepsis3 = TRUE
)
, ranked_admissions AS (
    SELECT
        sp.*,
        ie.hadm_id,
        ie.intime,
        ie.outtime,
        ie.los,
        ROW_NUMBER() OVER (
            PARTITION BY sp.subject_id
            ORDER BY ie.intime
        ) AS admission_rank
    FROM sepsis_patients sp
    INNER JOIN `physionet-data.mimiciv_3_1_icu.icustays` ie
        ON sp.stay_id = ie.stay_id
)
SELECT *
FROM ranked_admissions
WHERE admission_rank = 1;

Query is running:   0%|          |

Downloading:   0%|          |

In [7]:
# Count patients after filtering
step2_count = len(step2_result)
step2_unique_patients = step2_result['subject_id'].nunique()
step2_unique_stays = step2_result['stay_id'].nunique()

# Calculate exclusions
excluded_step2 = step1_unique_stays - step2_unique_stays

print("="*70)
print("STEP 2: First ICU Admission with Sepsis")
print("="*70)
print(f"Remaining ICU stays: {step2_unique_stays:,}")
print(f"Unique patients: {step2_unique_patients:,}")
print(f"\n❌ Excluded (not first admission): {excluded_step2:,}")
print("\n📊 Flow Diagram Update:")
print(f"   Started with: {step1_unique_stays:,} ICU stays")
print(f"   After first admission filter: {step2_unique_stays:,} ICU stays")
print(f"   Excluded: {excluded_step2:,} ICU stays")
print("="*70)

# Show sample
print("\n📋 Sample data (first 5 rows):")
print(step2_result[['subject_id', 'stay_id', 'hadm_id', 'intime', 'admission_rank', 'sofa_score']].head())

STEP 2: First ICU Admission with Sepsis
Remaining ICU stays: 33,311
Unique patients: 33,311

❌ Excluded (not first admission): 10,394

📊 Flow Diagram Update:
   Started with: 43,705 ICU stays
   After first admission filter: 33,311 ICU stays
   Excluded: 10,394 ICU stays

📋 Sample data (first 5 rows):
   subject_id   stay_id   hadm_id              intime  admission_rank  \
0    10236942  35201523  21834040 2127-03-08 03:31:17               1   
1    19494483  35007902  21352163 2177-05-03 19:42:08               1   
2    13916620  30652743  29961119 2175-07-13 12:51:38               1   
3    18021132  33981102  28026398 2160-10-19 00:21:35               1   
4    15102238  33907606  26923425 2129-09-14 10:37:06               1   

   sofa_score  
0          15  
1          13  
2          13  
3          12  
4          13  


---

## STEP 3: Apply Age Filter (≥18 years)

Exclude patients younger than 18 years old at ICU admission.

**Why this matters:**
- Adult sepsis pathophysiology differs from pediatric
- Treatment protocols are age-specific
- Most clinical trials exclude pediatric patients

**Method:**
- Use `patients` table with `anchor_age`
- Filter for `anchor_age >= 18`

**Note:** MIMIC-IV uses anchor_age for privacy protection (ages >89 are set to 91)

In [10]:
%%bigquery step3_result

WITH first_sepsis_stays AS (
    SELECT
        s.stay_id,
        s.subject_id,
        ROW_NUMBER() OVER (PARTITION BY s.subject_id ORDER BY ie.intime) AS rn
    FROM `my-new-project-473015.my_mimiciv_derived.sepsis3` s
    INNER JOIN `physionet-data.mimiciv_3_1_icu.icustays` ie
        ON s.stay_id = ie.stay_id
    WHERE s.sepsis3 = TRUE
)
, step2_data AS (
    SELECT
        s.subject_id,
        s.stay_id,
        s.suspected_infection_time,
        s.sofa_time,
        s.sofa_score,
        s.respiration,
        s.coagulation,
        s.liver,
        s.cardiovascular,
        s.cns,
        s.renal,
        ie.hadm_id,
        ie.intime,
        ie.outtime,
        ie.los
    FROM `my-new-project-473015.my_mimiciv_derived.sepsis3` s
    INNER JOIN `physionet-data.mimiciv_3_1_icu.icustays` ie
        ON s.stay_id = ie.stay_id
    INNER JOIN first_sepsis_stays fs
        ON s.stay_id = fs.stay_id
    WHERE s.sepsis3 = TRUE
        AND fs.rn = 1
)
SELECT
    sd.*,
    p.anchor_age,
    p.gender
FROM step2_data sd
INNER JOIN `physionet-data.mimiciv_3_1_hosp.patients` p
    ON sd.subject_id = p.subject_id
WHERE p.anchor_age >= 18;

Query is running:   0%|          |

Downloading:   0%|          |

In [12]:
# Count patients after age filter
step3_count = len(step3_result)
step3_unique_patients = step3_result['subject_id'].nunique()
step3_unique_stays = step3_result['stay_id'].nunique()

# Calculate exclusions
excluded_step3 = step2_unique_stays - step3_unique_stays

print("="*70)
print("STEP 3: Age Filter (≥18 years)")
print("="*70)
print(f"Remaining ICU stays: {step3_unique_stays:,}")
print(f"Unique patients: {step3_unique_patients:,}")
print(f"\n❌ Excluded (age <18): {excluded_step3:,}")
print(f"\nAge statistics:")
print(f"   Mean age: {step3_result['anchor_age'].mean():.1f} years")
print(f"   Median age: {step3_result['anchor_age'].median():.1f} years")
print(f"   Min age: {step3_result['anchor_age'].min()} years")
print(f"   Max age: {step3_result['anchor_age'].max()} years")
print("\n📊 Flow Diagram Update:")
print(f"   After first admission filter: {step2_unique_stays:,} ICU stays")
print(f"   After age ≥18 filter: {step3_unique_stays:,} ICU stays")
print(f"   Excluded: {excluded_step3:,} ICU stays")
print("="*70)

# Show sample
print("\n📋 Sample data (first 5 rows):")
print(step3_result[['subject_id', 'stay_id', 'anchor_age', 'gender', 'sofa_score', 'los']].head())

STEP 3: Age Filter (≥18 years)
Remaining ICU stays: 33,311
Unique patients: 33,311

❌ Excluded (age <18): 0

Age statistics:
   Mean age: 64.9 years
   Median age: 66.0 years
   Min age: 18 years
   Max age: 91 years

📊 Flow Diagram Update:
   After first admission filter: 33,311 ICU stays
   After age ≥18 filter: 33,311 ICU stays
   Excluded: 0 ICU stays

📋 Sample data (first 5 rows):
   subject_id   stay_id  anchor_age gender  sofa_score        los
0    15301569  33337491          24      M           3  16.670185
1    12021623  33274622          32      F           4  25.338958
2    16989568  30344574          39      M           6  11.719410
3    10670236  30992197          32      M           3  18.950567
4    10963753  31539056          32      M           2   3.664259


---

### 💡 Why Zero Pediatric Patients?

**Result:** 0 patients excluded for age <18

**This is EXPECTED and CORRECT because:**

1. **MIMIC-IV is an Adult-Only ICU Database (v2.2+)**
   - From MIMIC-IV v2.2 release notes:
     > *"Neonates have been removed from the dataset. Neonatal data will be released in a separate project with data from the neonatal intensive care unit."*
   - Beth Israel Deaconess Medical Center adult ICUs only
   - Pediatric and neonatal patients are excluded from MIMIC-IV
   - Minimum age in MIMIC-IV v3.1 is 18 years

2. **Separate Databases for Young Patients**
   - Neonatal data: Released separately as a dedicated neonatal database
   - Pediatric ICU data: Not included in MIMIC-IV
   - This adult-only focus ensures appropriate cohort selection

3. **Research Implications**
   - We can confidently state our cohort is "adult patients (≥18 years)"
   - No need to adjust for pediatric/neonatal confounders
   - Results cannot be generalized to children or neonates
   - Age filter step validates database integrity

**For the Flow Diagram:**
- We still include this step for transparency and reproducibility
- Shows we explicitly verified age criteria
- Documents systematic cohort selection process
- Required for publication standards (STROBE guidelines)

**Conclusion:** Zero exclusions validates we're using MIMIC-IV v3.1 correctly, which contains only adult ICU patients by design.

---

## STEP 4: Exclude Short ICU Stays (<24 hours)

Exclude patients with ICU length of stay <24 hours.

**Rationale:**
- Very short ICU stays may represent:
  - Transfer patients (moved to another facility)
  - Early deaths
  - Misclassification (not truly ICU-level care)
- 24-hour minimum allows observation of clinical trajectory
- Common exclusion criterion in sepsis research

**Two Methods to Calculate LOS:**

### Method A: Direct LOS column
- Uses `icustays.los` (stored as days)
- Already calculated in the database
- Convert to hours: `los * 24`

### Method B: Calculate from timestamps
- Uses `DATETIME_DIFF(outtime, intime, HOUR)`
- Calculated directly from admission/discharge times
- May reveal data quality issues (e.g., negative values)

**We will compare both methods to check for:**
1. ⚠️ Negative LOS values (outtime before intime - data error)
2. ⚠️ Discrepancies between methods
3. ⚠️ Number of patients excluded by each method

In [13]:
%%bigquery step4_comparison

WITH first_sepsis_stays AS (
    SELECT
        s.stay_id,
        s.subject_id,
        ROW_NUMBER() OVER (PARTITION BY s.subject_id ORDER BY ie.intime) AS rn
    FROM `my-new-project-473015.my_mimiciv_derived.sepsis3` s
    INNER JOIN `physionet-data.mimiciv_3_1_icu.icustays` ie
        ON s.stay_id = ie.stay_id
    WHERE s.sepsis3 = TRUE
)
, step3_data AS (
    SELECT
        s.subject_id,
        s.stay_id,
        s.suspected_infection_time,
        s.sofa_time,
        s.sofa_score,
        s.respiration,
        s.coagulation,
        s.liver,
        s.cardiovascular,
        s.cns,
        s.renal,
        ie.hadm_id,
        ie.intime,
        ie.outtime,
        ie.los,
        p.anchor_age,
        p.gender
    FROM `my-new-project-473015.my_mimiciv_derived.sepsis3` s
    INNER JOIN `physionet-data.mimiciv_3_1_icu.icustays` ie
        ON s.stay_id = ie.stay_id
    INNER JOIN first_sepsis_stays fs
        ON s.stay_id = fs.stay_id
    INNER JOIN `physionet-data.mimiciv_3_1_hosp.patients` p
        ON s.subject_id = p.subject_id
    WHERE s.sepsis3 = TRUE
        AND fs.rn = 1
        AND p.anchor_age >= 18
)
SELECT
    *,
    -- Method A: Direct LOS (days to hours)
    los * 24 AS los_hours_method_a,
    -- Method B: Calculate from timestamps
    DATETIME_DIFF(outtime, intime, HOUR) AS los_hours_method_b,
    -- Difference between methods
    (los * 24) - DATETIME_DIFF(outtime, intime, HOUR) AS los_difference
FROM step3_data;

Query is running:   0%|          |

Downloading:   0%|          |

In [14]:
# Analyze LOS calculation methods
print("="*70)
print("STEP 4: LOS Calculation Method Comparison")
print("="*70)

# Check for negative LOS values
negative_method_a = len(step4_comparison[step4_comparison['los_hours_method_a'] < 0])
negative_method_b = len(step4_comparison[step4_comparison['los_hours_method_b'] < 0])

print(f"\n⚠️  Data Quality Check:")
print(f"   Negative LOS (Method A - direct los*24): {negative_method_a}")
print(f"   Negative LOS (Method B - timestamp diff): {negative_method_b}")

if negative_method_b > 0:
    print(f"\n   ⚠️ WARNING: {negative_method_b} patients have outtime BEFORE intime!")
    print("   This indicates data quality issues in the timestamp fields.")
    print("\n   Sample of negative LOS cases:")
    print(step4_comparison[step4_comparison['los_hours_method_b'] < 0][
        ['stay_id', 'intime', 'outtime', 'los_hours_method_b']
    ].head())

# Compare methods
print(f"\n📊 LOS Statistics:")
print(f"   Method A (los*24) - Mean: {step4_comparison['los_hours_method_a'].mean():.1f} hours")
print(f"   Method B (timestamp) - Mean: {step4_comparison['los_hours_method_b'].mean():.1f} hours")
print(f"   Difference - Mean: {step4_comparison['los_difference'].mean():.2f} hours")
print(f"   Difference - Std: {step4_comparison['los_difference'].std():.2f} hours")

# Check for discrepancies
large_discrepancy = len(step4_comparison[abs(step4_comparison['los_difference']) > 1])
print(f"\n   Cases with >1 hour discrepancy: {large_discrepancy}")

# Count exclusions by each method
excluded_method_a = len(step4_comparison[step4_comparison['los_hours_method_a'] < 24])
excluded_method_b = len(step4_comparison[step4_comparison['los_hours_method_b'] < 24])

print(f"\n❌ Exclusions (<24 hours):")
print(f"   Method A (los*24): {excluded_method_a} patients")
print(f"   Method B (timestamp): {excluded_method_b} patients")
print(f"   Difference: {abs(excluded_method_a - excluded_method_b)} patients")

print("="*70)

STEP 4: LOS Calculation Method Comparison

⚠️  Data Quality Check:
   Negative LOS (Method A - direct los*24): 0
   Negative LOS (Method B - timestamp diff): 0

📊 LOS Statistics:
   Method A (los*24) - Mean: 127.9 hours
   Method B (timestamp) - Mean: 127.9 hours
   Difference - Mean: 0.00 hours
   Difference - Std: 0.41 hours

   Cases with >1 hour discrepancy: 0

❌ Exclusions (<24 hours):
   Method A (los*24): 3093 patients
   Method B (timestamp): 2885 patients
   Difference: 208 patients


In [15]:
print("="*70)
print("STEP 4c: Detailed Analysis of 208-Patient Difference")
print("="*70)

# Identify patients near the 24-hour boundary
boundary_analysis = step4_comparison.copy()
boundary_analysis['method_a_exclude'] = boundary_analysis['los_hours_method_a'] < 24
boundary_analysis['method_b_exclude'] = boundary_analysis['los_hours_method_b'] < 24

# Find discordant cases (excluded by one method but not the other)
discordant = boundary_analysis[
    boundary_analysis['method_a_exclude'] != boundary_analysis['method_b_exclude']
]

print(f"\n🔍 Discordant Cases (excluded by one method but not the other):")
print(f"   Total discordant: {len(discordant)} patients")

if len(discordant) > 0:
    # Analyze the discordant cases
    print(f"\n   Excluded by Method A only: {len(discordant[discordant['method_a_exclude'] == True])} patients")
    print(f"   Excluded by Method B only: {len(discordant[discordant['method_b_exclude'] == True])} patients")

    print(f"\n   LOS range for discordant cases:")
    print(f"      Method A: {discordant['los_hours_method_a'].min():.2f} - {discordant['los_hours_method_a'].max():.2f} hours")
    print(f"      Method B: {discordant['los_hours_method_b'].min():.2f} - {discordant['los_hours_method_b'].max():.2f} hours")
    print(f"      Difference: {discordant['los_difference'].min():.2f} - {discordant['los_difference'].max():.2f} hours")

    print(f"\n📋 Sample of discordant cases (showing boundary cases):")
    print(discordant[['stay_id', 'los_hours_method_a', 'los_hours_method_b', 'los_difference']].head(10))

# Patients near 24-hour boundary (23-25 hours)
near_boundary = step4_comparison[
    ((step4_comparison['los_hours_method_a'] >= 23) & (step4_comparison['los_hours_method_a'] <= 25)) |
    ((step4_comparison['los_hours_method_b'] >= 23) & (step4_comparison['los_hours_method_b'] <= 25))
]

print(f"\n📊 Patients near 24-hour boundary (23-25 hours):")
print(f"   Total: {len(near_boundary)} patients")
print(f"   This represents the 'sensitive zone' for the exclusion criterion")

print("\n" + "="*70)
print("💡 Recommendation:")
print("="*70)
print("Use Method A (direct los column) because:")
print("   1. It's the official MIMIC-IV calculated LOS")
print("   2. Accounts for any database-specific logic")
print("   3. Standard practice in MIMIC-IV research")
print("   4. Both methods show similar overall statistics")
print(f"\n   Final exclusions using Method A: {excluded_method_a} patients (<24 hours)")
print("="*70)

STEP 4c: Detailed Analysis of 208-Patient Difference

🔍 Discordant Cases (excluded by one method but not the other):
   Total discordant: 208 patients

   Excluded by Method A only: 208 patients
   Excluded by Method B only: 0 patients

   LOS range for discordant cases:
      Method A: 23.06 - 24.00 hours
      Method B: 24.00 - 24.00 hours
      Difference: -0.94 - -0.00 hours

📋 Sample of discordant cases (showing boundary cases):
       stay_id  los_hours_method_a  los_hours_method_b  los_difference
174   39717790           23.939167                  24       -0.060833
442   30462740           23.860278                  24       -0.139722
1438  39641919           23.913611                  24       -0.086389
1831  39086781           23.616389                  24       -0.383611
1856  32145696           23.116667                  24       -0.883333
2097  33701414           23.608056                  24       -0.391944
2171  31603687           23.898611                  24       -0.1

---

### 🔬 Key Finding: 208-Patient Discrepancy Explained

**What we discovered:**

1. **All 208 patients were excluded ONLY by Method A**
   - Method A (los column): 23.06 - 24.00 hours
   - Method B (timestamp): Exactly 24.00 hours for all
   - Difference: -0.94 to -0.00 hours

2. **Why does this happen?**
   - **Method A (los column)**: Stored as DAYS with high precision
     - Calculated: `los * 24` preserves decimal precision
     - Example: 0.9974 days → 23.94 hours
   - **Method B (timestamp)**: Uses `DATETIME_DIFF(outtime, intime, HOUR)`
     - Rounds to nearest hour boundary
     - Example: 23.94 hours → 24 hours (rounded up)

3. **These are "boundary cases"**
   - All 208 patients had LOS very close to 24 hours
   - Small rounding differences determine inclusion/exclusion
   - 1,226 total patients in the "sensitive zone" (23-25 hours)

**Decision: Use Method A (los column)**

**Rationale:**
- ✅ More precise (preserves sub-hour accuracy)
- ✅ Official MIMIC-IV calculation method
- ✅ Standard practice in published research
- ✅ Conservative approach (excludes borderline cases)
- ✅ Transparent and reproducible

**For Flow Diagram:**
- **Excluded for LOS <24 hours: 3,093 patients** (using Method A)

In [16]:
%%bigquery step4_result

WITH first_sepsis_stays AS (
    SELECT
        s.stay_id,
        s.subject_id,
        ROW_NUMBER() OVER (PARTITION BY s.subject_id ORDER BY ie.intime) AS rn
    FROM `my-new-project-473015.my_mimiciv_derived.sepsis3` s
    INNER JOIN `physionet-data.mimiciv_3_1_icu.icustays` ie
        ON s.stay_id = ie.stay_id
    WHERE s.sepsis3 = TRUE
)
, step3_data AS (
    SELECT
        s.subject_id,
        s.stay_id,
        s.suspected_infection_time,
        s.sofa_time,
        s.sofa_score,
        s.respiration,
        s.coagulation,
        s.liver,
        s.cardiovascular,
        s.cns,
        s.renal,
        ie.hadm_id,
        ie.intime,
        ie.outtime,
        ie.los,
        p.anchor_age,
        p.gender
    FROM `my-new-project-473015.my_mimiciv_derived.sepsis3` s
    INNER JOIN `physionet-data.mimiciv_3_1_icu.icustays` ie
        ON s.stay_id = ie.stay_id
    INNER JOIN first_sepsis_stays fs
        ON s.stay_id = fs.stay_id
    INNER JOIN `physionet-data.mimiciv_3_1_hosp.patients` p
        ON s.subject_id = p.subject_id
    WHERE s.sepsis3 = TRUE
        AND fs.rn = 1
        AND p.anchor_age >= 18
)
SELECT
    *,
    los * 24 AS los_hours
FROM step3_data
WHERE los * 24 >= 24;

Query is running:   0%|          |

Downloading:   0%|          |

In [17]:
# Count final cohort after LOS filter
step4_count = len(step4_result)
step4_unique_patients = step4_result['subject_id'].nunique()
step4_unique_stays = step4_result['stay_id'].nunique()

# Calculate exclusions
excluded_step4 = step3_unique_stays - step4_unique_stays

print("="*70)
print("STEP 4: Final Results - LOS ≥24 hours (Method A)")
print("="*70)
print(f"Remaining ICU stays: {step4_unique_stays:,}")
print(f"Unique patients: {step4_unique_patients:,}")
print(f"\n❌ Excluded (LOS <24 hours): {excluded_step4:,}")
print(f"\nLOS Statistics (hours):")
print(f"   Mean: {step4_result['los_hours'].mean():.1f} hours ({step4_result['los_hours'].mean()/24:.1f} days)")
print(f"   Median: {step4_result['los_hours'].median():.1f} hours ({step4_result['los_hours'].median()/24:.1f} days)")
print(f"   Min: {step4_result['los_hours'].min():.1f} hours")
print(f"   Max: {step4_result['los_hours'].max():.1f} hours")
print("\n📊 Flow Diagram Update:")
print(f"   After age ≥18 filter: {step3_unique_stays:,} ICU stays")
print(f"   After LOS ≥24h filter: {step4_unique_stays:,} ICU stays")
print(f"   Excluded: {excluded_step4:,} ICU stays")
print("="*70)

# Show sample
print("\n📋 Sample data (first 5 rows):")
print(step4_result[['subject_id', 'stay_id', 'anchor_age', 'gender', 'sofa_score', 'los_hours']].head())

STEP 4: Final Results - LOS ≥24 hours (Method A)
Remaining ICU stays: 30,218
Unique patients: 30,218

❌ Excluded (LOS <24 hours): 3,093

LOS Statistics (hours):
   Mean: 139.1 hours (5.8 days)
   Median: 76.8 hours (3.2 days)
   Min: 24.0 hours
   Max: 3832.0 hours

📊 Flow Diagram Update:
   After age ≥18 filter: 33,311 ICU stays
   After LOS ≥24h filter: 30,218 ICU stays
   Excluded: 3,093 ICU stays

📋 Sample data (first 5 rows):
   subject_id   stay_id  anchor_age gender  sofa_score   los_hours
0    15301569  33337491          24      M           3  400.084444
1    12021623  33274622          32      F           4  608.135000
2    16989568  30344574          39      M           6  281.265833
3    10670236  30992197          32      M           3  454.813611
4    10963753  31539056          32      M           2   87.942222


---

## STEP 5: Create Final Cohort Table

Save the final cohort to a permanent table for future analysis.

**Final Cohort Criteria:**
✅ Sepsis-3 diagnosis (SOFA ≥2 + suspected infection)
✅ First ICU admission with sepsis
✅ Age ≥18 years
✅ ICU length of stay ≥24 hours

**Table Name:** `my_mimiciv_derived.sepsis3_cohort`

**What we save:**
- Patient demographics (subject_id, age, gender)
- ICU stay information (stay_id, hadm_id, times, LOS)
- Sepsis onset details (infection time, SOFA time)
- SOFA scores and components
- Additional clinical data for analysis

In [18]:
%%bigquery

CREATE OR REPLACE TABLE `my-new-project-473015.my_mimiciv_derived.sepsis3_cohort` AS
WITH first_sepsis_stays AS (
    SELECT
        s.stay_id,
        s.subject_id,
        ROW_NUMBER() OVER (PARTITION BY s.subject_id ORDER BY ie.intime) AS rn
    FROM `my-new-project-473015.my_mimiciv_derived.sepsis3` s
    INNER JOIN `physionet-data.mimiciv_3_1_icu.icustays` ie
        ON s.stay_id = ie.stay_id
    WHERE s.sepsis3 = TRUE
)
SELECT
    -- Patient identifiers
    s.subject_id,
    s.stay_id,
    ie.hadm_id,

    -- Demographics
    p.anchor_age,
    p.gender,

    -- ICU stay times
    ie.intime AS icu_intime,
    ie.outtime AS icu_outtime,
    ie.los AS los_days,
    ie.los * 24 AS los_hours,

    -- Sepsis onset information
    s.suspected_infection_time,
    s.sofa_time,
    DATETIME_DIFF(s.suspected_infection_time, ie.intime, HOUR) AS infection_time_from_icu_admit_hours,

    -- SOFA scores
    s.sofa_score AS sofa_at_sepsis_onset,
    s.respiration AS sofa_respiration,
    s.coagulation AS sofa_coagulation,
    s.liver AS sofa_liver,
    s.cardiovascular AS sofa_cardiovascular,
    s.cns AS sofa_cns,
    s.renal AS sofa_renal,

    -- Inclusion/exclusion tracking
    'Included' AS cohort_status,
    CURRENT_TIMESTAMP() AS cohort_creation_time

FROM `my-new-project-473015.my_mimiciv_derived.sepsis3` s
INNER JOIN `physionet-data.mimiciv_3_1_icu.icustays` ie
    ON s.stay_id = ie.stay_id
INNER JOIN first_sepsis_stays fs
    ON s.stay_id = fs.stay_id
INNER JOIN `physionet-data.mimiciv_3_1_hosp.patients` p
    ON s.subject_id = p.subject_id
WHERE s.sepsis3 = TRUE
    AND fs.rn = 1
    AND p.anchor_age >= 18
    AND ie.los * 24 >= 24;

Query is running:   0%|          |

""


In [19]:
%%bigquery final_cohort_check

SELECT
    COUNT(*) AS total_patients,
    COUNT(DISTINCT subject_id) AS unique_subjects,
    COUNT(DISTINCT stay_id) AS unique_stays,
    COUNT(DISTINCT hadm_id) AS unique_admissions,
    ROUND(AVG(anchor_age), 1) AS mean_age,
    ROUND(AVG(sofa_at_sepsis_onset), 2) AS mean_sofa,
    ROUND(AVG(los_hours), 1) AS mean_los_hours
FROM `my-new-project-473015.my_mimiciv_derived.sepsis3_cohort`;

Query is running:   0%|          |

Downloading:   0%|          |

In [20]:
print("="*70)
print("STEP 5: Final Cohort Table Created Successfully")
print("="*70)
print(f"Table: my-new-project-473015.my_mimiciv_derived.sepsis3_cohort")
print(f"\nTotal patients: {final_cohort_check['total_patients'][0]:,}")
print(f"Unique subjects: {final_cohort_check['unique_subjects'][0]:,}")
print(f"Unique ICU stays: {final_cohort_check['unique_stays'][0]:,}")
print(f"Unique hospital admissions: {final_cohort_check['unique_admissions'][0]:,}")
print(f"\nMean age: {final_cohort_check['mean_age'][0]:.1f} years")
print(f"Mean SOFA at onset: {final_cohort_check['mean_sofa'][0]:.2f}")
print(f"Mean LOS: {final_cohort_check['mean_los_hours'][0]:.1f} hours ({final_cohort_check['mean_los_hours'][0]/24:.1f} days)")
print("="*70)
print("\n✅ Final cohort table is ready for analysis!")

STEP 5: Final Cohort Table Created Successfully
Table: my-new-project-473015.my_mimiciv_derived.sepsis3_cohort

Total patients: 30,218
Unique subjects: 30,218
Unique ICU stays: 30,218
Unique hospital admissions: 30,218

Mean age: 64.8 years
Mean SOFA at onset: 3.65
Mean LOS: 139.1 hours (5.8 days)

✅ Final cohort table is ready for analysis!


---

## STEP 6: Flow Diagram & Cohort Statistics

Create a comprehensive flow diagram showing patient selection at each step.

**Flow diagrams are essential for:**
- Transparency in cohort selection
- Meeting STROBE (Strengthening the Reporting of Observational Studies in Epidemiology) guidelines
- Publication requirements in medical journals
- Reproducibility of research
- Quality control and validation

We will create:
1. **Numerical flow diagram** (text-based)
2. **Summary statistics** of the final cohort
3. **Exclusion summary table**

In [22]:
print("="*80)
print(" " * 20 + "SEPSIS-3 COHORT SELECTION FLOW DIAGRAM")
print("="*80)
print()
print("┌─────────────────────────────────────────────────────────────────────────┐")
print("│                                                                         │")
print("│  MIMIC-IV v3.1 Database: All Sepsis-3 Patients                         │")
print(f"│  N = {step1_unique_stays:,} ICU stays                                          │")
print("│  (Sepsis-3 criteria: SOFA ≥2 + suspected infection)                    │")
print("│                                                                         │")
print("└─────────────────────────────────────────────────────────────────────────┘")
print("                                  │")
print("                                  │")
print("                                  ▼")
print("┌─────────────────────────────────────────────────────────────────────────┐")
print("│  STEP 1: Extract all Sepsis-3 patients from sepsis3 table              │")
print(f"│  Remaining: {step1_unique_stays:,} ICU stays                                   │")
print("└─────────────────────────────────────────────────────────────────────────┘")
print("                                  │")
print("                                  │")
print("                                  ▼")
print("┌─────────────────────────────────────────────────────────────────────────┐")
print("│  STEP 2: Filter for FIRST ICU admission with sepsis                    │")
print(f"│  Remaining: {step2_unique_stays:,} ICU stays                                   │")
excluded_step2_display = step1_unique_stays - step2_unique_stays
print(f"│  Excluded: {excluded_step2_display:,} (not first admission)                              │")
print("└─────────────────────────────────────────────────────────────────────────┘")
print("                                  │")
print("                                  │")
print("                                  ▼")
print("┌─────────────────────────────────────────────────────────────────────────┐")
print("│  STEP 3: Apply age filter (≥18 years)                                  │")
print(f"│  Remaining: {step3_unique_stays:,} ICU stays                                   │")
excluded_step3_display = step2_unique_stays - step3_unique_stays
print(f"│  Excluded: {excluded_step3_display:,} (age <18 years)                                     │")
print("└─────────────────────────────────────────────────────────────────────────┘")
print("                                  │")
print("                                  │")
print("                                  ▼")
print("┌─────────────────────────────────────────────────────────────────────────┐")
print("│  STEP 4: Exclude short ICU stays (<24 hours)                           │")
print(f"│  Remaining: {step4_unique_stays:,} ICU stays                                   │")
excluded_step4_display = step3_unique_stays - step4_unique_stays
print(f"│  Excluded: {excluded_step4_display:,} (LOS <24 hours)                              │")
print("└─────────────────────────────────────────────────────────────────────────┘")
print("                                  │")
print("                                  │")
print("                                  ▼")
print("╔═════════════════════════════════════════════════════════════════════════╗")
print("║                                                                         ║")
print("║                    FINAL SEPSIS-3 COHORT                                ║")
print(f"║                    N = {step4_unique_stays:,} patients                              ║")
print("║                                                                         ║")
print("╚═════════════════════════════════════════════════════════════════════════╝")
print()
print("="*80)

                    SEPSIS-3 COHORT SELECTION FLOW DIAGRAM

┌─────────────────────────────────────────────────────────────────────────┐
│                                                                         │
│  MIMIC-IV v3.1 Database: All Sepsis-3 Patients                         │
│  N = 43,705 ICU stays                                          │
│  (Sepsis-3 criteria: SOFA ≥2 + suspected infection)                    │
│                                                                         │
└─────────────────────────────────────────────────────────────────────────┘
                                  │
                                  │
                                  ▼
┌─────────────────────────────────────────────────────────────────────────┐
│  STEP 1: Extract all Sepsis-3 patients from sepsis3 table              │
│  Remaining: 43,705 ICU stays                                   │
└─────────────────────────────────────────────────────────────────────────┘
                 

In [23]:
import pandas as pd

# Create exclusion summary
exclusion_data = {
    'Step': [
        'Starting population',
        'After first admission filter',
        'After age ≥18 filter',
        'After LOS ≥24h filter',
        'FINAL COHORT'
    ],
    'Remaining (N)': [
        f"{step1_unique_stays:,}",
        f"{step2_unique_stays:,}",
        f"{step3_unique_stays:,}",
        f"{step4_unique_stays:,}",
        f"{step4_unique_stays:,}"
    ],
    'Excluded (N)': [
        '-',
        f"{step1_unique_stays - step2_unique_stays:,}",
        f"{step2_unique_stays - step3_unique_stays:,}",
        f"{step3_unique_stays - step4_unique_stays:,}",
        '-'
    ],
    'Exclusion Reason': [
        'All Sepsis-3 patients',
        'Not first ICU admission',
        'Age <18 years',
        'ICU LOS <24 hours',
        'Met all criteria'
    ]
}

exclusion_df = pd.DataFrame(exclusion_data)

print("="*80)
print("EXCLUSION SUMMARY TABLE")
print("="*80)
print(exclusion_df.to_string(index=False))
print("="*80)

EXCLUSION SUMMARY TABLE
                        Step Remaining (N) Excluded (N)        Exclusion Reason
         Starting population        43,705            -   All Sepsis-3 patients
After first admission filter        33,311       10,394 Not first ICU admission
        After age ≥18 filter        33,311            0           Age <18 years
       After LOS ≥24h filter        30,218        3,093       ICU LOS <24 hours
                FINAL COHORT        30,218            -        Met all criteria
